# Time Series Analysis Fundamentals - ML Interview Prep

This notebook covers essential concepts for ML interviews:
- Time series fundamentals and components
- Stationarity and transformation techniques
- ARIMA and SARIMA modeling
- Practical implementation with Air Passengers dataset
- Interview practice questions

---
# Part 1: Theory
---

## 1.1 What is Time Series Data?

### Definition

A **time series** is a sequence of data points collected at successive, equally spaced points in time.

### Time Series vs Cross-Sectional Data

| Aspect | Time Series | Cross-Sectional |
|--------|-------------|----------------|
| **Structure** | Observations over time | Observations at a single point in time |
| **Order** | Order matters (temporal dependency) | Order doesn't matter |
| **Independence** | Usually NOT independent | Usually assumed independent |
| **Examples** | Stock prices, temperature, sales | Survey responses, patient data |

### Key Characteristics

1. **Temporal Ordering**: The sequence of observations is meaningful
2. **Autocorrelation**: Past values influence future values
3. **Patterns**: May exhibit trend, seasonality, or cycles

## 1.2 Components of Time Series

A time series $Y_t$ can be decomposed into:

### 1. Trend ($T_t$)
- Long-term increase or decrease in the data
- Example: Population growth, technology adoption

### 2. Seasonality ($S_t$)
- Regular, predictable patterns that repeat at fixed intervals
- Example: Holiday sales, summer temperatures

### 3. Residual/Noise ($R_t$)
- Random fluctuations not explained by trend or seasonality
- Should be stationary if model is well-specified

### Decomposition Models

| Model | Formula | When to Use |
|-------|---------|-------------|
| **Additive** | $Y_t = T_t + S_t + R_t$ | Seasonal variation is constant over time |
| **Multiplicative** | $Y_t = T_t \times S_t \times R_t$ | Seasonal variation grows with the level of the series |

## 1.3 Stationarity

### Why Stationarity Matters

Most time series models (ARIMA, ARMA) assume the data is **stationary**. Non-stationary data must be transformed before modeling.

### Definition

A time series is **stationary** if its statistical properties don't change over time:

| Property | Stationary | Non-Stationary |
|----------|------------|----------------|
| Mean | Constant | Changes over time |
| Variance | Constant | Changes over time |
| Autocorrelation | Only depends on lag | Depends on time |

### Types of Non-Stationarity

1. **Trend**: Mean changes over time
2. **Seasonality**: Periodic fluctuations
3. **Heteroscedasticity**: Variance changes over time

### Testing for Stationarity

**1. Visual Inspection**
- Plot the series: look for trends, changing variance
- Check rolling mean and standard deviation

**2. Augmented Dickey-Fuller (ADF) Test**
- $H_0$: Series has a unit root (non-stationary)
- $H_1$: Series is stationary
- If p-value < 0.05, reject $H_0$ → series is stationary

### Making Data Stationary

| Technique | Addresses | Method |
|-----------|-----------|--------|
| **Differencing** | Trend | $Y'_t = Y_t - Y_{t-1}$ |
| **Seasonal Differencing** | Seasonality | $Y'_t = Y_t - Y_{t-s}$ |
| **Log Transform** | Heteroscedasticity | $Y'_t = \log(Y_t)$ |

## 1.4 ACF and PACF

### Autocorrelation Function (ACF)

Measures the correlation between a time series and its lagged versions.

$$\text{ACF}(k) = \text{Corr}(Y_t, Y_{t-k})$$

- Includes both direct and indirect correlations
- Used to identify MA order (q)

### Partial Autocorrelation Function (PACF)

Measures the correlation between $Y_t$ and $Y_{t-k}$ after removing the effects of intermediate lags.

- Isolates direct relationship at each lag
- Used to identify AR order (p)

### Using ACF/PACF to Choose ARIMA Parameters

| Pattern | ACF | PACF | Model |
|---------|-----|------|-------|
| AR(p) | Tails off (gradual decay) | Cuts off after lag p | Use PACF to find p |
| MA(q) | Cuts off after lag q | Tails off | Use ACF to find q |
| ARMA(p,q) | Tails off | Tails off | Try different combinations |

**"Cuts off"** = Sharp drop to near zero

**"Tails off"** = Gradual decay toward zero

## 1.5 ARIMA Model

### Components: ARIMA(p, d, q)

| Parameter | Name | Meaning |
|-----------|------|--------|
| **p** | AR order | Number of autoregressive terms (lags of Y) |
| **d** | Differencing | Number of times data is differenced |
| **q** | MA order | Number of moving average terms (lags of error) |

### Mathematical Formulation

**AR(p)**: $Y_t = c + \phi_1 Y_{t-1} + \phi_2 Y_{t-2} + ... + \phi_p Y_{t-p} + \epsilon_t$

**MA(q)**: $Y_t = \mu + \epsilon_t + \theta_1 \epsilon_{t-1} + ... + \theta_q \epsilon_{t-q}$

**ARIMA(p,d,q)**: Combines AR and MA on differenced data

### Special Cases

| Model | Parameters | Description |
|-------|------------|-------------|
| ARIMA(0,0,0) | White noise | Random fluctuations |
| ARIMA(1,0,0) | AR(1) | First-order autoregression |
| ARIMA(0,0,1) | MA(1) | First-order moving average |
| ARIMA(0,1,0) | Random walk | Non-stationary, differenced = white noise |

## 1.6 SARIMA for Seasonal Data

### SARIMA(p, d, q)(P, D, Q, s)

Extends ARIMA to handle seasonality with additional parameters:

| Parameter | Meaning |
|-----------|--------|
| P | Seasonal AR order |
| D | Seasonal differencing |
| Q | Seasonal MA order |
| s | Seasonal period (e.g., 12 for monthly data with yearly seasonality) |

### Example

SARIMA(1, 1, 1)(1, 1, 1, 12) for monthly data:
- Non-seasonal: AR(1), differencing(1), MA(1)
- Seasonal: AR(1), differencing(1), MA(1) with period 12

---
# Part 2: Implementation
---

## 2.1 Environment Setup & Data Loading

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

# Time series specific
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Evaluation
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Settings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Environment setup complete!")

In [ ]:
def load_air_passengers(data_dir: Path = Path('data')) -> pd.Series:
    """
    Load the Air Passengers dataset.
    Downloads from GitHub if not present locally.
    """
    data_dir.mkdir(exist_ok=True)
    filepath = data_dir / 'air_passengers.csv'
    
    if not filepath.exists():
        print("Downloading Air Passengers dataset...")
        url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
        df = pd.read_csv(url)
        df.to_csv(filepath, index=False)
        print(f"Dataset saved to {filepath}")
    else:
        print(f"Loading dataset from {filepath}")
        df = pd.read_csv(filepath)
    
    # Convert to proper time series format
    df['Month'] = pd.to_datetime(df['Month'])
    df.set_index('Month', inplace=True)
    
    # Return as Series
    series = df['Passengers']
    series.index.freq = 'MS'  # Month Start frequency
    
    return series

# Load data
passengers = load_air_passengers()
print(f"\nDataset shape: {passengers.shape}")
print(f"Date range: {passengers.index.min()} to {passengers.index.max()}")
passengers.head(10)

## 2.2 Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print("=" * 50)
print("DATASET STATISTICS")
print("=" * 50)
print(passengers.describe())
print(f"\nMissing values: {passengers.isnull().sum()}")

In [ ]:
# Plot the time series
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Original series
axes[0, 0].plot(passengers, color='blue', linewidth=1.5)
axes[0, 0].set_title('Air Passengers (1949-1960)')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Number of Passengers (thousands)')

# Distribution
axes[0, 1].hist(passengers, bins=20, edgecolor='black', alpha=0.7)
axes[0, 1].axvline(passengers.mean(), color='red', linestyle='--', label=f'Mean: {passengers.mean():.1f}')
axes[0, 1].set_title('Distribution of Passengers')
axes[0, 1].set_xlabel('Passengers')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].legend()

# Box plot by year
passengers_df = passengers.to_frame()
passengers_df['Year'] = passengers_df.index.year
passengers_df.boxplot(column='Passengers', by='Year', ax=axes[1, 0])
axes[1, 0].set_title('Passengers by Year')
axes[1, 0].set_xlabel('Year')
axes[1, 0].set_ylabel('Passengers')
plt.suptitle('')  # Remove auto title

# Box plot by month
passengers_df['Month_Num'] = passengers_df.index.month
passengers_df.boxplot(column='Passengers', by='Month_Num', ax=axes[1, 1])
axes[1, 1].set_title('Passengers by Month')
axes[1, 1].set_xlabel('Month')
axes[1, 1].set_ylabel('Passengers')
plt.suptitle('')  # Remove auto title

plt.tight_layout()
plt.show()

In [ ]:
# Rolling statistics to visualize trend
rolling_mean = passengers.rolling(window=12).mean()
rolling_std = passengers.rolling(window=12).std()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rolling mean
axes[0].plot(passengers, label='Original', alpha=0.7)
axes[0].plot(rolling_mean, label='12-Month Rolling Mean', color='red', linewidth=2)
axes[0].set_title('Rolling Mean (12 months)')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Passengers')
axes[0].legend()

# Rolling std
axes[1].plot(rolling_std, label='12-Month Rolling Std', color='green', linewidth=2)
axes[1].set_title('Rolling Standard Deviation (12 months)')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Standard Deviation')
axes[1].legend()

plt.tight_layout()
plt.show()

print("Observations:")
print("- Rolling mean shows upward TREND")
print("- Rolling std increases over time (heteroscedasticity)")
print("- This suggests multiplicative decomposition may be appropriate")

## 2.3 Time Series Decomposition

In [ ]:
# Additive decomposition
decomposition_add = seasonal_decompose(passengers, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(12, 10))

axes[0].plot(passengers)
axes[0].set_title('Original Series')
axes[0].set_ylabel('Passengers')

axes[1].plot(decomposition_add.trend)
axes[1].set_title('Trend Component')
axes[1].set_ylabel('Trend')

axes[2].plot(decomposition_add.seasonal)
axes[2].set_title('Seasonal Component')
axes[2].set_ylabel('Seasonal')

axes[3].plot(decomposition_add.resid)
axes[3].set_title('Residual Component')
axes[3].set_ylabel('Residual')

fig.suptitle('Additive Decomposition', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Multiplicative decomposition (more appropriate for this data)
decomposition_mult = seasonal_decompose(passengers, model='multiplicative', period=12)

fig, axes = plt.subplots(4, 1, figsize=(12, 10))

axes[0].plot(passengers)
axes[0].set_title('Original Series')
axes[0].set_ylabel('Passengers')

axes[1].plot(decomposition_mult.trend)
axes[1].set_title('Trend Component')
axes[1].set_ylabel('Trend')

axes[2].plot(decomposition_mult.seasonal)
axes[2].set_title('Seasonal Component')
axes[2].set_ylabel('Seasonal')

axes[3].plot(decomposition_mult.resid)
axes[3].set_title('Residual Component')
axes[3].set_ylabel('Residual')

fig.suptitle('Multiplicative Decomposition', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Note: Multiplicative decomposition is more appropriate here because:")
print("- The seasonal variation increases with the level of the series")
print("- The residuals appear more homoscedastic (constant variance)")

## 2.4 Stationarity Testing and Transformation

In [ ]:
def adf_test(series, title=''):
    """
    Perform Augmented Dickey-Fuller test and print results.
    """
    result = adfuller(series.dropna())
    
    print(f"ADF Test Results: {title}")
    print("-" * 50)
    print(f"ADF Statistic: {result[0]:.4f}")
    print(f"p-value: {result[1]:.4f}")
    print(f"Lags Used: {result[2]}")
    print(f"Number of Observations: {result[3]}")
    print("Critical Values:")
    for key, value in result[4].items():
        print(f"   {key}: {value:.4f}")
    
    if result[1] < 0.05:
        print("\nConclusion: STATIONARY (reject H0)")
        return True
    else:
        print("\nConclusion: NON-STATIONARY (fail to reject H0)")
        return False

In [ ]:
# Test original series
print("=" * 60)
is_stationary = adf_test(passengers, 'Original Series')
print("=" * 60)

In [ ]:
# Apply transformations to achieve stationarity
fig, axes = plt.subplots(3, 2, figsize=(14, 12))

# 1. Log transformation (stabilize variance)
passengers_log = np.log(passengers)
axes[0, 0].plot(passengers_log)
axes[0, 0].set_title('Log Transformed')
axes[0, 0].set_ylabel('Log(Passengers)')

# ADF test
axes[0, 1].text(0.5, 0.5, f"ADF p-value: {adfuller(passengers_log)[1]:.4f}\n\nStill non-stationary", 
                ha='center', va='center', fontsize=14, transform=axes[0, 1].transAxes)
axes[0, 1].set_title('ADF Test Result')
axes[0, 1].axis('off')

# 2. First differencing (remove trend)
passengers_diff = passengers_log.diff().dropna()
axes[1, 0].plot(passengers_diff)
axes[1, 0].set_title('Log + First Difference')
axes[1, 0].set_ylabel('Diff(Log(Passengers))')

# ADF test
adf_result = adfuller(passengers_diff)
axes[1, 1].text(0.5, 0.5, f"ADF p-value: {adf_result[1]:.4f}\n\n{'Stationary!' if adf_result[1] < 0.05 else 'Still non-stationary'}", 
                ha='center', va='center', fontsize=14, transform=axes[1, 1].transAxes)
axes[1, 1].set_title('ADF Test Result')
axes[1, 1].axis('off')

# 3. Seasonal differencing (remove seasonality)
passengers_seasonal_diff = passengers_log.diff(12).dropna()
axes[2, 0].plot(passengers_seasonal_diff)
axes[2, 0].set_title('Log + Seasonal Difference (lag=12)')
axes[2, 0].set_ylabel('Seasonal Diff')

# ADF test
adf_result_seasonal = adfuller(passengers_seasonal_diff)
axes[2, 1].text(0.5, 0.5, f"ADF p-value: {adf_result_seasonal[1]:.4f}\n\n{'Stationary!' if adf_result_seasonal[1] < 0.05 else 'Still non-stationary'}", 
                ha='center', va='center', fontsize=14, transform=axes[2, 1].transAxes)
axes[2, 1].set_title('ADF Test Result')
axes[2, 1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Final transformation: Log + First Diff + Seasonal Diff
passengers_stationary = passengers_log.diff().diff(12).dropna()

plt.figure(figsize=(12, 4))
plt.plot(passengers_stationary)
plt.title('Stationary Series (Log + First Diff + Seasonal Diff)')
plt.xlabel('Date')
plt.ylabel('Value')
plt.axhline(y=0, color='r', linestyle='--', alpha=0.5)
plt.show()

print("=" * 60)
adf_test(passengers_stationary, 'Fully Transformed Series')
print("=" * 60)

## 2.5 ACF and PACF Analysis

In [ ]:
# ACF and PACF plots for parameter selection
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# First differenced series (for non-seasonal parameters)
plot_acf(passengers_diff, ax=axes[0, 0], lags=40, alpha=0.05)
axes[0, 0].set_title('ACF - First Differenced Series')

plot_pacf(passengers_diff, ax=axes[0, 1], lags=40, alpha=0.05, method='ywm')
axes[0, 1].set_title('PACF - First Differenced Series')

# Fully differenced series (stationary)
plot_acf(passengers_stationary, ax=axes[1, 0], lags=40, alpha=0.05)
axes[1, 0].set_title('ACF - Stationary Series')

plot_pacf(passengers_stationary, ax=axes[1, 1], lags=40, alpha=0.05, method='ywm')
axes[1, 1].set_title('PACF - Stationary Series')

plt.tight_layout()
plt.show()

print("ACF/PACF Interpretation:")
print("-" * 50)
print("- Significant spike at lag 12 indicates seasonality")
print("- PACF cuts off after lag 1 -> suggests AR(1)")
print("- ACF shows gradual decay -> supports AR component")
print("\nSuggested starting point: SARIMA(1,1,1)(1,1,1,12)")

## 2.6 Simple Forecasting Methods

In [ ]:
# Train/Test split (time series style - last 24 months for testing)
train_size = len(passengers) - 24
train = passengers[:train_size]
test = passengers[train_size:]

print(f"Training set: {train.index.min()} to {train.index.max()} ({len(train)} observations)")
print(f"Test set: {test.index.min()} to {test.index.max()} ({len(test)} observations)")

In [ ]:
def evaluate_forecast(y_true, y_pred, model_name="Model"):
    """
    Calculate and return forecast evaluation metrics.
    """
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    
    return {
        'Model': model_name,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE (%)': mape
    }

In [ ]:
# 1. Simple Moving Average
def moving_average_forecast(train, test, window=12):
    """
    Forecast using simple moving average.
    """
    # Use last 'window' observations from train for forecast
    forecast = [train[-window:].mean()] * len(test)
    return pd.Series(forecast, index=test.index)

ma_forecast = moving_average_forecast(train, test, window=12)
ma_metrics = evaluate_forecast(test, ma_forecast, "Simple Moving Average")

print("Simple Moving Average Results:")
print("-" * 40)
for key, value in ma_metrics.items():
    if key != 'Model':
        print(f"{key}: {value:.2f}")

In [ ]:
# 2. Exponential Smoothing (Holt-Winters)
hw_model = ExponentialSmoothing(
    train,
    seasonal_periods=12,
    trend='add',
    seasonal='mul'  # Multiplicative seasonality
).fit()

hw_forecast = hw_model.forecast(len(test))
hw_metrics = evaluate_forecast(test, hw_forecast, "Holt-Winters")

print("Holt-Winters Exponential Smoothing Results:")
print("-" * 40)
for key, value in hw_metrics.items():
    if key != 'Model':
        print(f"{key}: {value:.2f}")

In [ ]:
# Visualize simple forecasts
plt.figure(figsize=(14, 6))
plt.plot(train, label='Train', color='blue')
plt.plot(test, label='Test (Actual)', color='black', linewidth=2)
plt.plot(ma_forecast, label='Moving Average', color='orange', linestyle='--')
plt.plot(hw_forecast, label='Holt-Winters', color='green', linestyle='--')
plt.title('Simple Forecasting Methods Comparison')
plt.xlabel('Date')
plt.ylabel('Passengers')
plt.legend()
plt.show()

## 2.7 ARIMA Modeling

In [ ]:
# Fit ARIMA model (non-seasonal)
# Based on ACF/PACF analysis: p=1, d=1, q=1

arima_model = ARIMA(train, order=(1, 1, 1)).fit()

print("ARIMA(1,1,1) Model Summary")
print("=" * 50)
print(arima_model.summary().tables[1])

In [ ]:
# ARIMA forecast
arima_forecast = arima_model.forecast(steps=len(test))
arima_metrics = evaluate_forecast(test, arima_forecast, "ARIMA(1,1,1)")

print("ARIMA(1,1,1) Results:")
print("-" * 40)
for key, value in arima_metrics.items():
    if key != 'Model':
        print(f"{key}: {value:.2f}")

print("\nNote: ARIMA doesn't capture seasonality, so performance is limited")

In [ ]:
# Model diagnostics
fig = arima_model.plot_diagnostics(figsize=(14, 10))
plt.suptitle('ARIMA(1,1,1) Diagnostic Plots', y=1.02)
plt.tight_layout()
plt.show()

print("Diagnostic Interpretation:")
print("-" * 50)
print("- Standardized residuals: Should look like white noise")
print("- Histogram: Should be approximately normal")
print("- Normal Q-Q: Points should follow the diagonal line")
print("- Correlogram: No significant autocorrelations in residuals")

## 2.8 SARIMA Modeling

In [ ]:
# Fit SARIMA model
# SARIMA(p,d,q)(P,D,Q,s)
# Based on analysis: (1,1,1)(1,1,1,12)

sarima_model = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

print("SARIMA(1,1,1)(1,1,1,12) Model Summary")
print("=" * 60)
print(sarima_model.summary().tables[1])

In [ ]:
# SARIMA forecast
sarima_forecast = sarima_model.forecast(steps=len(test))
sarima_metrics = evaluate_forecast(test, sarima_forecast, "SARIMA(1,1,1)(1,1,1,12)")

print("SARIMA(1,1,1)(1,1,1,12) Results:")
print("-" * 40)
for key, value in sarima_metrics.items():
    if key != 'Model':
        print(f"{key}: {value:.2f}")

print("\nSARIMA captures seasonality, significantly improving performance!")

In [ ]:
# SARIMA diagnostics
fig = sarima_model.plot_diagnostics(figsize=(14, 10))
plt.suptitle('SARIMA(1,1,1)(1,1,1,12) Diagnostic Plots', y=1.02)
plt.tight_layout()
plt.show()

## 2.9 Model Comparison and Final Forecast

In [ ]:
# Collect all metrics
all_metrics = [ma_metrics, hw_metrics, arima_metrics, sarima_metrics]
comparison_df = pd.DataFrame(all_metrics)
comparison_df = comparison_df.set_index('Model')

print("Model Comparison:")
print("=" * 70)
print(comparison_df.round(2).to_string())
print("\nBest model: SARIMA (lowest MAPE)")

In [ ]:
# Visual comparison of all forecasts
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Full view
axes[0].plot(train, label='Train', color='blue', alpha=0.7)
axes[0].plot(test, label='Test (Actual)', color='black', linewidth=2)
axes[0].plot(ma_forecast, label='Moving Average', linestyle='--', alpha=0.7)
axes[0].plot(hw_forecast, label='Holt-Winters', linestyle='--', alpha=0.7)
axes[0].plot(arima_forecast, label='ARIMA', linestyle='--', alpha=0.7)
axes[0].plot(sarima_forecast, label='SARIMA', linestyle='--', alpha=0.7)
axes[0].set_title('All Models Comparison')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Passengers')
axes[0].legend(loc='upper left')

# Zoomed test period
axes[1].plot(test, label='Actual', color='black', linewidth=2, marker='o', markersize=4)
axes[1].plot(ma_forecast, label=f'MA (MAPE={ma_metrics["MAPE (%)"]:.1f}%)', linestyle='--', marker='s', markersize=3)
axes[1].plot(hw_forecast, label=f'HW (MAPE={hw_metrics["MAPE (%)"]:.1f}%)', linestyle='--', marker='^', markersize=3)
axes[1].plot(arima_forecast, label=f'ARIMA (MAPE={arima_metrics["MAPE (%)"]:.1f}%)', linestyle='--', marker='d', markersize=3)
axes[1].plot(sarima_forecast, label=f'SARIMA (MAPE={sarima_metrics["MAPE (%)"]:.1f}%)', linestyle='--', marker='v', markersize=3)
axes[1].set_title('Test Period (Zoomed)')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Passengers')
axes[1].legend(loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# Metrics bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

models = comparison_df.index.tolist()
x_pos = np.arange(len(models))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

# RMSE
axes[0].bar(x_pos, comparison_df['RMSE'], color=colors, alpha=0.8)
axes[0].set_ylabel('RMSE')
axes[0].set_title('RMSE Comparison (lower is better)')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(models, rotation=20, ha='right')

# MAE
axes[1].bar(x_pos, comparison_df['MAE'], color=colors, alpha=0.8)
axes[1].set_ylabel('MAE')
axes[1].set_title('MAE Comparison (lower is better)')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(models, rotation=20, ha='right')

# MAPE
axes[2].bar(x_pos, comparison_df['MAPE (%)'], color=colors, alpha=0.8)
axes[2].set_ylabel('MAPE (%)')
axes[2].set_title('MAPE Comparison (lower is better)')
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(models, rotation=20, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# Future forecast with best model (SARIMA)
# Refit on full data
final_model = SARIMAX(
    passengers,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

# Forecast next 24 months
future_forecast = final_model.get_forecast(steps=24)
forecast_mean = future_forecast.predicted_mean
forecast_ci = future_forecast.conf_int(alpha=0.05)

# Plot
plt.figure(figsize=(14, 6))
plt.plot(passengers, label='Historical Data', color='blue')
plt.plot(forecast_mean, label='Forecast', color='red', linewidth=2)
plt.fill_between(forecast_ci.index, 
                 forecast_ci.iloc[:, 0], 
                 forecast_ci.iloc[:, 1], 
                 color='red', alpha=0.2, label='95% Confidence Interval')
plt.title('Air Passengers Forecast (Next 24 Months)')
plt.xlabel('Date')
plt.ylabel('Passengers (thousands)')
plt.legend()
plt.show()

print("\nForecast Summary (Next 24 Months):")
print("-" * 40)
print(f"Forecast Start: {forecast_mean.index.min()}")
print(f"Forecast End: {forecast_mean.index.max()}")
print(f"Min Forecast: {forecast_mean.min():.1f}")
print(f"Max Forecast: {forecast_mean.max():.1f}")

---
# Part 3: Interview Quiz
---

Test your understanding with these common interview questions. Click to reveal answers.

### Q1: What is stationarity and why is it important for time series modeling?

<details>
<summary>Click to reveal answer</summary>

**Stationarity** means the statistical properties of a time series (mean, variance, autocorrelation) are constant over time.

**Why it matters:**
- Most time series models (ARIMA, ARMA) assume stationarity
- Non-stationary data can lead to spurious relationships and unreliable forecasts
- Model parameters become meaningful and stable only for stationary data

**Types:**
- **Strict stationarity**: All moments are time-invariant (rare in practice)
- **Weak stationarity**: Mean, variance, and autocovariance are time-invariant (what we typically test for)

**How to test:**
- Visual inspection (rolling mean/std)
- Augmented Dickey-Fuller (ADF) test
- KPSS test
</details>

### Q2: What do p, d, and q represent in ARIMA(p, d, q)?

<details>
<summary>Click to reveal answer</summary>

| Parameter | Name | Meaning | How to Choose |
|-----------|------|---------|---------------|
| **p** | AR order | Number of autoregressive lags (past values of Y) | PACF cuts off after lag p |
| **d** | Differencing | Number of times to difference the data | Until series is stationary (ADF test) |
| **q** | MA order | Number of moving average lags (past errors) | ACF cuts off after lag q |

**Example: ARIMA(2, 1, 1)**
- p=2: Use last 2 values ($Y_{t-1}$, $Y_{t-2}$)
- d=1: Difference once to make stationary
- q=1: Use last 1 error term ($\epsilon_{t-1}$)

**Mathematical form:**
$$Y'_t = c + \phi_1 Y'_{t-1} + \phi_2 Y'_{t-2} + \theta_1 \epsilon_{t-1} + \epsilon_t$$
where $Y'_t = Y_t - Y_{t-1}$ (differenced)
</details>

### Q3: How do you choose ARIMA parameters using ACF and PACF?

<details>
<summary>Click to reveal answer</summary>

**Step-by-step approach:**

1. **Check stationarity** first. If not stationary, difference the data (determines d)

2. **Look at ACF and PACF** of the stationary series:

| Pattern | ACF | PACF | Model |
|---------|-----|------|-------|
| AR(p) | Tails off (exponential/sinusoidal decay) | Cuts off after lag p | Use PACF to find p |
| MA(q) | Cuts off after lag q | Tails off | Use ACF to find q |
| ARMA(p,q) | Tails off | Tails off | Try combinations |

3. **"Cuts off"** = Drops sharply to near zero and stays within confidence bands

4. **"Tails off"** = Decays gradually toward zero

**Alternative: Use AIC/BIC** to compare different parameter combinations and select the model with lowest information criterion.
</details>

### Q4: What is the difference between ACF and PACF?

<details>
<summary>Click to reveal answer</summary>

**ACF (Autocorrelation Function):**
- Measures correlation between $Y_t$ and $Y_{t-k}$
- Includes **both direct and indirect** effects through intermediate lags
- Example: ACF(3) includes effect of lag 1 and 2 as well

**PACF (Partial Autocorrelation Function):**
- Measures correlation between $Y_t$ and $Y_{t-k}$ **after removing** effects of intermediate lags
- Shows only the **direct** relationship at each lag
- Example: PACF(3) shows correlation at lag 3 only, controlling for lags 1 and 2

**Analogy:**
- ACF: Total correlation including all paths
- PACF: Direct correlation only, like partial regression coefficient

**Use:**
- ACF → identify MA order (q)
- PACF → identify AR order (p)
</details>

### Q5: How is time series cross-validation different from regular cross-validation?

<details>
<summary>Click to reveal answer</summary>

**Regular k-fold CV:**
- Randomly splits data into k folds
- Any observation can be in train or test
- Assumes observations are independent

**Time Series CV (Walk-Forward Validation):**
- **Respects temporal order** - train always before test
- Never uses future data to predict past
- Expanding or sliding window approach

**Example (Expanding Window):**
```
Fold 1: Train [1-10], Test [11-12]
Fold 2: Train [1-12], Test [13-14]
Fold 3: Train [1-14], Test [15-16]
...
```

**Why it matters:**
- Time series has temporal dependencies
- Random shuffling would leak future information (data leakage)
- More realistic simulation of real-world forecasting

**In sklearn:** Use `TimeSeriesSplit` instead of `KFold`
</details>

### Q6: How do you handle seasonality in time series?

<details>
<summary>Click to reveal answer</summary>

**1. Decomposition:**
- Additive: $Y_t = T_t + S_t + R_t$ (constant seasonal amplitude)
- Multiplicative: $Y_t = T_t \times S_t \times R_t$ (seasonal amplitude grows with level)

**2. Seasonal Differencing:**
- $Y'_t = Y_t - Y_{t-s}$ where s is the seasonal period
- Example: For monthly data with yearly seasonality, s=12

**3. SARIMA:**
- Extends ARIMA with seasonal terms: (P, D, Q, s)
- Models both seasonal and non-seasonal patterns

**4. Fourier Terms:**
- Add sine and cosine terms as features
- Useful with machine learning models

**5. Seasonal Dummy Variables:**
- One-hot encode months/quarters
- Works with any regression model

**How to identify seasonality:**
- Box plots by period (month, quarter)
- ACF shows significant spikes at seasonal lags
- Seasonal decomposition
</details>

### Q7: What is the difference between additive and multiplicative decomposition?

<details>
<summary>Click to reveal answer</summary>

**Additive Decomposition:**
$$Y_t = T_t + S_t + R_t$$

- Seasonal variation is **constant** regardless of level
- Example: Sales increase by $1000 every December (same absolute amount)
- Use when: Seasonal fluctuations don't change with the mean

**Multiplicative Decomposition:**
$$Y_t = T_t \times S_t \times R_t$$

- Seasonal variation **scales with the level**
- Example: Sales increase by 20% every December (proportional)
- Use when: Seasonal amplitude grows as the series grows

**How to choose:**
- Plot the data: if seasonal swings grow over time → multiplicative
- Log transform converts multiplicative to additive:
  $\log(Y_t) = \log(T_t) + \log(S_t) + \log(R_t)$
- Check residual variance: should be more constant with correct model
</details>

### Q8: What metrics are used to evaluate time series forecasts?

<details>
<summary>Click to reveal answer</summary>

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **MAE** | $\frac{1}{n}\sum|y_t - \hat{y}_t|$ | Average absolute error, same units as data |
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_t - \hat{y}_t)^2}$ | Penalizes large errors more heavily |
| **MAPE** | $\frac{100}{n}\sum|\frac{y_t - \hat{y}_t}{y_t}|$ | Percentage error, scale-independent |
| **sMAPE** | $\frac{200}{n}\sum\frac{|y_t - \hat{y}_t|}{|y_t| + |\hat{y}_t|}$ | Symmetric MAPE, handles zeros better |

**Choosing metrics:**
- **MAE**: Robust to outliers, interpretable
- **RMSE**: Differentiable, penalizes large errors
- **MAPE**: Good for comparing across different scales
  - Caution: Undefined when actual = 0
  - Asymmetric: penalizes over-forecasts less

**Best practice:** Report multiple metrics for complete picture.
</details>

### Q9: How can overfitting manifest in time series models?

<details>
<summary>Click to reveal answer</summary>

**Signs of overfitting:**
1. **Training vs test gap**: Low in-sample error, high out-of-sample error
2. **Too many parameters**: High p and q values relative to data size
3. **Near-perfect fit**: Unrealistically low residuals on training data

**Causes:**
- Including too many AR/MA terms
- Fitting to noise rather than signal
- Short training period with complex model

**Prevention strategies:**
1. **Use information criteria** (AIC, BIC) to penalize complexity
2. **Time series CV** with out-of-sample validation
3. **Parsimony**: Start simple, add complexity only if needed
4. **Check residuals**: Should be white noise (no patterns left)

**Rule of thumb:** Total parameters (p + q + P + Q) should be much smaller than sample size.
</details>

### Q10: Explain the Augmented Dickey-Fuller (ADF) test.

<details>
<summary>Click to reveal answer</summary>

**Purpose:** Test for stationarity (specifically, unit root)

**Hypotheses:**
- $H_0$: Series has a unit root (non-stationary)
- $H_1$: Series is stationary

**Test equation:**
$$\Delta Y_t = \alpha + \beta t + \gamma Y_{t-1} + \sum_{i=1}^{p} \delta_i \Delta Y_{t-i} + \epsilon_t$$

- Tests if $\gamma = 0$ (unit root present)
- Augmented with lagged differences to handle autocorrelation

**Interpretation:**
- **p-value < 0.05**: Reject $H_0$ → Series is stationary
- **p-value ≥ 0.05**: Fail to reject $H_0$ → Series is non-stationary

**In practice:**
```python
from statsmodels.tsa.stattools import adfuller
result = adfuller(series)
print(f'ADF Statistic: {result[0]}')
print(f'p-value: {result[1]}')
```
</details>

### Q11: What is exponential smoothing and when would you use it?

<details>
<summary>Click to reveal answer</summary>

**Concept:** Weight recent observations more heavily than older ones, with weights decaying exponentially.

**Types:**

| Model | Components | Use Case |
|-------|------------|----------|
| Simple ES | Level only | No trend, no seasonality |
| Holt's | Level + Trend | Trend, no seasonality |
| Holt-Winters | Level + Trend + Seasonality | Trend and seasonality |

**Simple Exponential Smoothing:**
$$\hat{Y}_{t+1} = \alpha Y_t + (1-\alpha) \hat{Y}_t$$
- $\alpha$: Smoothing parameter (0 < α < 1)
- Higher α → more weight to recent data

**When to use:**
- Quick, interpretable forecasts
- Short-term forecasting
- When computational efficiency matters
- As a benchmark before trying complex models

**Advantages over ARIMA:**
- Simpler, fewer parameters
- No stationarity requirement
- Often works well in practice
</details>

### Q12: What is the difference between AR and MA processes?

<details>
<summary>Click to reveal answer</summary>

**AR (Autoregressive) Process:**
$$Y_t = c + \phi_1 Y_{t-1} + \phi_2 Y_{t-2} + ... + \epsilon_t$$

- Current value depends on **past values** of the series
- Memory through observations
- PACF cuts off, ACF tails off

**MA (Moving Average) Process:**
$$Y_t = \mu + \epsilon_t + \theta_1 \epsilon_{t-1} + \theta_2 \epsilon_{t-2} + ...$$

- Current value depends on **past errors** (shocks)
- Memory through error terms
- ACF cuts off, PACF tails off

**Comparison:**

| Aspect | AR | MA |
|--------|----|----|  
| Memory | Through past values | Through past shocks |
| Impulse response | Infinite | Finite (q periods) |
| ACF pattern | Decays gradually | Cuts off after lag q |
| PACF pattern | Cuts off after lag p | Decays gradually |

**Intuition:**
- AR: "The economy was good last month, so it's likely good this month"
- MA: "A shock happened 2 months ago that's still affecting us"
</details>

### Q13: How do you handle missing values in time series?

<details>
<summary>Click to reveal answer</summary>

**Methods:**

1. **Forward Fill (Last Observation Carried Forward):**
   - Use previous value to fill missing
   - `df.fillna(method='ffill')`
   - Good for: slowly changing series

2. **Backward Fill:**
   - Use next value to fill missing
   - `df.fillna(method='bfill')`
   
3. **Linear Interpolation:**
   - Estimate missing values linearly between neighbors
   - `df.interpolate(method='linear')`
   - Good for: trending data

4. **Seasonal Interpolation:**
   - Use same period from previous cycle
   - Good for: strongly seasonal data

5. **Model-based Imputation:**
   - Use ARIMA or other models to predict missing values
   - Most sophisticated but can introduce bias

**Best practices:**
- Understand why data is missing (MCAR, MAR, MNAR)
- Document imputation method
- Consider sensitivity analysis with different methods
- Don't impute if too many values missing
</details>

### Q14: What are the seasonal parameters (P, D, Q, s) in SARIMA?

<details>
<summary>Click to reveal answer</summary>

**SARIMA(p, d, q)(P, D, Q, s):**

| Parameter | Meaning | Example (monthly data, s=12) |
|-----------|---------|-----------------------------|
| **P** | Seasonal AR order | Lag 12, 24, ... |
| **D** | Seasonal differencing | $Y_t - Y_{t-12}$ |
| **Q** | Seasonal MA order | Errors at lag 12, 24, ... |
| **s** | Seasonal period | 12 for monthly, 4 for quarterly |

**Example: SARIMA(1,1,1)(1,1,1,12)**
- Non-seasonal: AR(1), difference once, MA(1)
- Seasonal: SAR(1) at lag 12, seasonal difference, SMA(1) at lag 12

**How to identify:**
- Look at ACF/PACF at seasonal lags (12, 24, 36...)
- If ACF has significant spike at lag 12 → consider seasonal component
- Seasonal differencing (D=1) usually sufficient for most data

**Common choices:**
- Monthly data: s=12
- Quarterly data: s=4
- Weekly data (daily obs): s=7
</details>

### Q15: When would you use ARIMA vs machine learning for time series?

<details>
<summary>Click to reveal answer</summary>

**Use ARIMA when:**
- Univariate time series (single variable)
- Focus on temporal patterns (trend, seasonality)
- Need interpretable models
- Limited data available
- Statistical inference needed (confidence intervals)

**Use ML (Random Forest, XGBoost, LSTM) when:**
- Multiple features/variables available
- Complex non-linear relationships
- Large datasets
- External predictors important (weather, holidays)
- Accuracy prioritized over interpretability

**Hybrid approaches:**
- Use ARIMA for baseline, beat it with ML
- Feature engineer time components (lag features, rolling stats)
- Ensemble: combine ARIMA and ML predictions

**Common ML time series features:**
- Lag values: $Y_{t-1}, Y_{t-2}, ...$
- Rolling statistics: moving average, std
- Time features: month, day of week, holiday
- External features: weather, economic indicators
</details>

---
## Summary

In this notebook, we covered:

1. **Time Series Fundamentals**: Definition, components (trend, seasonality, residual), and stationarity
2. **ACF/PACF Analysis**: How to interpret autocorrelation plots and select ARIMA parameters
3. **Practical Implementation**: Full pipeline from data loading to forecasting
4. **Model Comparison**: Simple methods vs ARIMA vs SARIMA
5. **Interview Prep**: 15 common questions with detailed answers

### Key Takeaways

- Always check for stationarity before fitting ARIMA models
- Use ACF/PACF to guide parameter selection
- SARIMA handles seasonality; use it for seasonal data
- Time series CV must respect temporal ordering
- Report multiple evaluation metrics (RMSE, MAE, MAPE)
- Start simple (exponential smoothing) before complex models